# Airbnb Price Prediction — Evaluation Notebook (CS6140)

**Purpose:** Reproducible evaluation of all trained models — no training is done here.

**Inputs (expected layout):**
- `data/engineered/features.csv` — train set after feature engineering (has `log_price`)
- `data/engineered/features_test.csv` — test set after feature engineering (no label)
- `data/raw/test.csv` — raw test set (only used to recover the `id` column for submissions)
- `saved/feature_names.joblib` — the final column order the models expect
- `saved/models/model_{linreg,xgb,histgb,linsvr,ann}.{joblib,keras}` — saved models
- `saved/models/scaler_ann.joblib` — StandardScaler fit during ANN training

**Outputs:**
- A validation metrics table (RMSE / R² / MAE) on a held-out 20% of the training data


## 1. Imports

In [15]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Preprocessing helper

Mirrors the preprocessing used during training so val and test data both end up aligned with `feature_names`.

In [16]:
FEATURE_NAMES = joblib.load('saved/feature_names.joblib')

def preprocess(df):
    """Apply the exact cleaning used during training, then align columns to FEATURE_NAMES."""
    df = df.copy()

    # Drop malformed amenity column (leading space)
    df = df.drop(columns=[c for c in df.columns if c.strip() == 'smooth pathway to front door'], errors='ignore')

    # host_response_rate: "100%" -> 100.0, NaN -> -1
    df['host_response_rate'] = (df['host_response_rate']
                                  .astype(str).str.rstrip('%')
                                  .replace('nan', np.nan)
                                  .astype(float)
                                  .fillna(-1))

    # Numeric NaN fills
    df['review_scores_rating'] = df['review_scores_rating'].fillna(-1)
    df['walkscore']            = df['walkscore'].fillna(df['walkscore'].median())
    df['transitscore']         = df['transitscore'].fillna(df['transitscore'].median())
    df['bathrooms']            = df['bathrooms'].fillna(0)
    df['beds']                 = df['beds'].fillna(0)
    df['bedrooms']             = df['bedrooms'].fillna(0)
    df['DateDiffHostSince']    = df['DateDiffHostSince'].fillna(-1)

    # One-hot encode categoricals
    df = pd.get_dummies(df,
        columns=['property_type','room_type','bed_type','cancellation_policy','city'],
        drop_first=False)

    # Bool -> int
    bool_cols = df.select_dtypes(include='bool').columns
    df[bool_cols] = df[bool_cols].astype(int)

    # Align columns (fills missing with 0, drops extras so shape matches training)
    df = df.reindex(columns=FEATURE_NAMES, fill_value=0)

    assert df.shape[1] == len(FEATURE_NAMES), "Column count mismatch vs. feature_names"
    assert df.isna().sum().sum() == 0, "NaNs remain after preprocessing"
    return df


## 3. Load & prepare validation split

We recreate the same 80/20 split used during training (`random_state=21`) so validation metrics are directly comparable.

In [17]:
train_df = pd.read_csv('data/engineered/features.csv')
y = train_df['log_price'].values
X_df = preprocess(train_df.drop(columns=['log_price']))
X = X_df.values

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=21)
print(f'X shape: {X.shape}   X_va shape: {X_va.shape}')


X shape: (74111, 206)   X_va shape: (14823, 206)


## 4. Load & prepare test set

`features_test.csv` has no `id` column, so we pull it from the raw test file.

In [18]:
test_fe   = pd.read_csv('data/engineered/features_test.csv')
test_raw  = pd.read_csv('data/raw/test.csv', usecols=['id'])

X_test_df = preprocess(test_fe)
X_test    = X_test_df.values
test_ids  = test_raw['id'].values
print(f'X_test shape: {X_test.shape}   #ids: {len(test_ids)}')
assert len(test_ids) == X_test.shape[0], "Row count mismatch between features_test and raw test"


X_test shape: (25458, 206)   #ids: 25458


## 5. Load all saved models

In [19]:
lr   = joblib.load('saved/models/model_linreg.joblib')
hgb  = joblib.load('saved/models/model_histgb.joblib')
svr  = joblib.load('saved/models/model_linsvr.joblib')

# XGBoost: refit inline on the full training matrix, then persist a fresh artifact.
# (Serialized xgboost files can deserialize incorrectly across xgboost/sklearn
# version boundaries, so we rebuild it here using the same hyperparameters
# used during training, then overwrite the saved joblib with the current binary.)
from xgboost import XGBRegressor
xgb = XGBRegressor(
    n_estimators=400, learning_rate=0.05, max_depth=6,
    n_jobs=-1, random_state=42, verbosity=0,
).fit(X, y)
joblib.dump(xgb, 'saved/models/model_xgb.joblib')

# ANN is a Keras model and needs its own scaler
from tensorflow.keras.models import load_model
ann     = load_model('saved/models/model_ann.keras', compile=False)
sc_ann  = joblib.load('saved/models/scaler_ann.joblib')

def ann_predict(Xin):
    return ann.predict(sc_ann.transform(Xin), verbose=0).ravel()

MODELS = {
    'LinearRegression': lambda Xin: lr.predict(Xin),
    'XGBoost':          lambda Xin: xgb.predict(Xin),
    'HistGradBoost':    lambda Xin: hgb.predict(Xin),
    'LinearSVR':        lambda Xin: svr.predict(Xin),
    'ANN':              ann_predict,
}
print('Loaded', len(MODELS), 'models')

C:\Users\vaibh\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\vaibh\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.6.1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\vaibh\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.6.1 

Loaded 5 models


## 6. Validation metrics (RMSE / R² / MAE)

Computed on the held-out 20% split from training data. 

In [20]:
rows = []
for name, predict in MODELS.items():
    pred = predict(X_va)
    mae  = mean_absolute_error(y_va, pred)
    mse  = mean_squared_error(y_va, pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_va, pred)
    rows.append({'model': name, 'RMSE': rmse, 'R2': r2, 'MAE': mae, 'MSE': mse})

results = pd.DataFrame(rows).sort_values('RMSE').reset_index(drop=True)
print(results.to_string(index=False, float_format=lambda x: f'{x:.4f}'))


           model   RMSE     R2    MAE    MSE
         XGBoost 0.3336 0.7835 0.2431 0.1113
   HistGradBoost 0.3454 0.7679 0.2517 0.1193
             ANN 0.3840 0.7132 0.2837 0.1474
LinearRegression 0.4346 0.6325 0.3224 0.1889
       LinearSVR 0.4414 0.6209 0.3204 0.1949


## 7. Test set predictions

`test.csv` has no labels (competition test set), so we cannot compute RMSE/MSE/MAE metrics.

In [21]:
for name, predict in MODELS.items():
    preds = predict(X_test)
    print(f'{name:18s}  mean={preds.mean():.3f}  std={preds.std():.3f} ')


LinearRegression    mean=4.781  std=0.572 
XGBoost             mean=4.778  std=0.615 
HistGradBoost       mean=4.778  std=0.614 
LinearSVR           mean=4.754  std=0.563 
ANN                 mean=4.793  std=0.628 
